# 02 - Piloto SFT corrigido
Três alvos supervisionados; não é um teste de emergência por supervisão exclusiva de saída. Use GPU e volume persistente. Código compilado, treino não executado na revisão.


In [ ]:
from pathlib import Path
import os, sys, json, subprocess
cwd = Path.cwd()
candidates = [cwd, cwd.parent, cwd / "revisao_2026", cwd.parent / "revisao_2026",
              cwd / "cot-obfuscation-h2a" / "revisao_2026"]
ROOT = next((p for p in candidates if (p / "src/instruments.py").is_file()), None)
if ROOT is None:
    clone = cwd / "cot-obfuscation-h2a"
    if not clone.exists():
        subprocess.run(["git", "clone", "https://github.com/Kelvin-Marques-Cyber/cot-obfuscation-h2a.git",
                        str(clone)], check=True)
    ROOT = clone / "revisao_2026"
assert (ROOT / "src/instruments.py").is_file(), ROOT
ROOT = ROOT.resolve()
sys.path.insert(0, str(ROOT / "src"))
os.chdir(ROOT)
ENV = dict(os.environ, PYTHONPATH=str(ROOT / "src") + os.pathsep + os.environ.get("PYTHONPATH", ""))
from notebook_runtime import run_logged

def run_step(command, log_name):
    # Atualiza credenciais/variáveis configuradas depois da célula inicial.
    current_env = dict(os.environ)
    current_env["PYTHONPATH"] = str(ROOT / "src") + os.pathsep + os.environ.get("PYTHONPATH", "")
    current_env["PYTHONUNBUFFERED"] = "1"
    return run_logged(command, ROOT / "logs" / log_name, cwd=ROOT, env=current_env)

print("Raiz:", ROOT)


In [ ]:
INSTALL_DEPS = False  # True na primeira instalação; depois reinicie o kernel se necessário.
if INSTALL_DEPS:
    import importlib.util
    if importlib.util.find_spec("torch") is None:
        raise RuntimeError("Instale primeiro PyTorch/CUDA para sua GPU: https://pytorch.org/get-started/locally/")
    run_step([sys.executable, "-m", "pip", "install", "-r", "requirements-gpu.txt"], "gpu_install.log")
    run_step([sys.executable, "-m", "pip", "check"], "gpu_pip_check.log")

import torch
assert torch.cuda.is_available(), "Selecione runtime GPU. Para CPU, use o notebook 01."
print("Torch:", torch.__version__, "CUDA:", torch.version.cuda)
print("GPU:", torch.cuda.get_device_name(0))
print("Capacidade:", torch.cuda.get_device_capability(0))
print("VRAM GiB:", round(torch.cuda.get_device_properties(0).total_memory/2**30, 2))
from importlib.metadata import version
assert version("trl") == "0.22.2", "Use a versão fixada e reinicie o kernel."
assert version("transformers") == "4.56.2", "Use a versão fixada e reinicie o kernel."


## Dados e plano
A mesma partição é compartilhada pelas condições e sementes de treino. Semente de dados e de treino são distintas. IDs evitam colisões entre partições.


In [ ]:
from instruments import make_disjoint_data
from gpu_lab.benchmark import write_dataset
from gpu_lab.train import validate_config
SMOKE = True
RUN_ROOT = (ROOT / "runs/sft").resolve()  # Troque para seu volume persistente.
DATA_PATH = ROOT / "data/sft_disjoint.jsonl"
data = make_disjoint_data(n_per_split=8 if SMOKE else 500, seed=20261015)
print(write_dataset(DATA_PATH, data))
template = json.loads((ROOT / "configs/sft_smoke.json").read_text())
plan = []
for seed in ([42] if SMOKE else [42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53]):
    for level in (["A"] if SMOKE else ["A", "B", "C"]):
        for condition in ["explicit", "direct", "generic"]:
            cfg = dict(template, train_seed=seed, level=level, condition=condition,
                       max_steps=4 if SMOKE else 200, output_root=str(RUN_ROOT))
            validate_config(cfg)
            plan.append(cfg)
print("Execuções planejadas:", len(plan))
print(json.dumps(plan[0], indent=2))


In [ ]:
EXECUTAR_TREINO = False  # Ative após conferir modelo, plano e diretório persistente.
if EXECUTAR_TREINO:
    RUN_ROOT.mkdir(parents=True, exist_ok=True)
    # Um plano existente é reutilizado, sem reconsultar a revisão mutável main.
    frozen_path = RUN_ROOT / "frozen_plan.json"
    if frozen_path.exists():
        stored = json.loads(frozen_path.read_text())
        strip_revision = lambda entries: [{k:v for k,v in c.items() if k != "revision"} for c in entries]
        assert strip_revision(stored) == strip_revision(plan), "Plano mudou: use novo RUN_ROOT."
        plan = stored
    else:
        from huggingface_hub import model_info
        shas = {(c["model_id"], c["revision"]): model_info(c["model_id"], revision=c["revision"]).sha for c in plan}
        for cfg in plan:
            cfg["revision"] = shas[(cfg["model_id"], cfg["revision"])]
        frozen_path.write_text(json.dumps(plan, indent=2))
    (RUN_ROOT / "pip-freeze.txt").write_text(subprocess.check_output(
        [sys.executable, "-m", "pip", "freeze"], text=True))
    for i, cfg in enumerate(plan):
        config_path = RUN_ROOT / f"config_{i:04d}.json"
        config_path.write_text(json.dumps(cfg, indent=2))
        print("Executando:", i+1, "/", len(plan), cfg["model_id"], cfg["condition"], cfg["train_seed"], flush=True)
        run_step([sys.executable, "-m", "gpu_lab.train", "--config", str(config_path),
                  "--data", str(DATA_PATH)], f"train_{i:04d}.log")
else:
    print("Plano preparado. Nenhum treino executado nesta célula.")


## Avaliar e retomar
A avaliação preserva enunciado, gabarito, saída integral, revisão dos pesos, tokens e motivo do término. O modo early usa o mesmo orçamento total, incluindo o prefixo forçado.


In [ ]:
EXECUTAR_AVALIACAO = False
if EXECUTAR_AVALIACAO:
    for status in sorted(RUN_ROOT.glob("*/status.json")):
        if json.loads(status.read_text()).get("status") == "trained":
            run_step([sys.executable, "-m", "gpu_lab.evaluate",
                "--run-dir", str(status.parent), "--data", str(DATA_PATH),
                "--budget", "384"], f"sft_eval_{status.parent.name}.log")
